# DOCtr KIE COR Extraction (modular)

This notebook is reorganized into clear cells: imports, model init, class definition, helper functions, and a run/save cell that writes extracted schedules to `extracted_courses_fixed.json` with keys `sched1`, `sched2`, ...

In [1]:
# Setup & Imports
from doctr.io import DocumentFile
from doctr.models import kie_predictor
import re
from typing import List, Dict
import json
import os


/home/sheer/Desktop/SchedScan/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Model Initialization (run this cell to (re)load the model)
print("Loading KIE predictor model...")
model = kie_predictor(
    det_arch='db_resnet50', 
    reco_arch='crnn_vgg16_bn',  # <-- REVERT THIS LINE
    pretrained=True
)
print("Model loaded successfully!")

Loading KIE predictor model...
Model loaded successfully!


In [3]:
class CORExtractor:
    def __init__(self, model):
        self.model = model

    def extract_from_document(self, file_path: str) -> List[Dict]:
        if file_path.lower().endswith('.pdf'):
            print("Loading PDF at 300 DPI (higher quality)...")
            scale_factor = 300 / 72  # Calculate scale for 300 DPI
            doc = DocumentFile.from_pdf(file_path, scale=scale_factor)
        else:
            doc = DocumentFile.from_images(file_path)
            
        result = self.model(doc)
        # We call the parser ONCE for the whole document
        return self._parse_document_elements(result)

    def _parse_document_elements(self, result) -> List[Dict]:
        all_elements = []
        for page in result.pages:
            predictions = page.predictions
            for class_name, pred_list in predictions.items():
                for pred in pred_list:
                    all_elements.append({
                        'text': pred.value.strip(),
                        'geometry': pred.geometry
                    })
        
        # Sort all elements by Y, then X. This is our master list.
        all_elements.sort(key=lambda x: (x['geometry'][0][1], x['geometry'][0][0]) if x['geometry'] else (0, 0))
        
        return self._group_into_courses(all_elements)

    def _group_into_courses(self, elements: List[Dict]) -> List[Dict]:
        """
        Geometric/Spatial parsing approach:
        1. Find anchors (subject codes)
        2. Find subject name spatially (to the right of anchor on same row)
        3. Collect details vertically below anchor (until next anchor)
        4. Parse details blob with regex
        """
        courses = []
        
        # --- PATTERNS ---
        subject_code_pattern = re.compile(r'^[A-Z]{4,}\d{5,}$')
        time_pattern = re.compile(r'([0-9]{2}:[0-9]{2}[AP]M)')
        day_pattern = re.compile(r'\b(M|T|W|TH|F|S|TF|MW|MWF|MTH|TTH)\b')
        # Robust location pattern that catches LR7, LAB2, CLA19, etc.
        location_pattern = re.compile(r'\b(LR\d*|LAB\d*|CLA\d*|COM\s?LAB\d*)\b', re.IGNORECASE)
        
        # Subject Name: "SE 131", "BAAE 9"
        internal_code_pattern = re.compile(r'^[A-Z]{2,5}\s?\d{1,3}$')
        
        # Numbers to ignore
        is_number_pattern = re.compile(r'^\d+(\.\d{1,2})?$')
        
        # Tolerances for geometric matching
        Y_TOLERANCE = 0.02  # Elements on same row if Y coords within this threshold
        
        # Find all subject code anchors
        anchors = []
        for i, elem in enumerate(elements):
            if subject_code_pattern.match(elem['text']):
                anchors.append({
                    'index': i,
                    'text': elem['text'],
                    'geometry': elem['geometry']
                })
        
        subject_name_map = {}  # Memory for repeated subject codes
        
        # Process each anchor
        for anchor_idx, anchor in enumerate(anchors):
            subject_code = anchor['text']
            anchor_geom = anchor['geometry']
            anchor_y = anchor_geom[0][1]  # min_y of anchor
            anchor_x_max = anchor_geom[1][0]  # max_x of anchor
            
            # Determine the range of elements for this anchor
            start_index = anchor['index']
            end_index = anchors[anchor_idx + 1]['index'] if anchor_idx + 1 < len(anchors) else len(elements)
            
            # --- STEP 1: Find Subject Name Spatially ---
            # Collect ALL text to the right of anchor on same row until we hit time/location/day
            current_subject_name = ''
            subject_name_parts = []
            
            for i in range(start_index + 1, end_index):
                elem = elements[i]
                elem_geom = elem['geometry']
                elem_y = elem_geom[0][1]
                elem_x = elem_geom[0][0]
                
                # Check if on same row (Y overlaps)
                if abs(elem_y - anchor_y) < Y_TOLERANCE:
                    # Check if to the right of anchor
                    if elem_x > anchor_x_max:
                        text = elem['text']
                        
                        # Stop collecting if we hit a time pattern (indicates schedule details)
                        if time_pattern.match(text):
                            break
                        
                        # Skip numbers, locations, and days
                        if is_number_pattern.match(text):
                            continue
                        if location_pattern.match(text):
                            break  # Location means we've passed the subject name
                        if day_pattern.match(text):
                            break  # Day means we've passed the subject name
                        
                        # Collect this text as part of subject name
                        subject_name_parts.append(text)
            
            # Join collected parts to form complete subject name
            if subject_name_parts:
                current_subject_name = ' '.join(subject_name_parts)
            
            # Fallback to memory if no name found
            if not current_subject_name:
                current_subject_name = subject_name_map.get(subject_code, '')
            else:
                subject_name_map[subject_code] = current_subject_name
            
            # --- STEP 2: Collect Details Blob (vertically below anchor) ---
            # All elements from anchor+1 to next anchor, excluding the subject name we just found
            details_blob = []
            for i in range(start_index + 1, end_index):
                elem = elements[i]
                # Skip the subject name element we already identified
                if elem['text'] != current_subject_name:
                    details_blob.append(elem['text'])
            
            # --- STEP 3: Parse Details Blob with Regex ---
            blob_text = ' '.join(details_blob)
            
            time_matches = time_pattern.findall(blob_text)
            day_matches = day_pattern.findall(blob_text)
            location_matches = location_pattern.findall(blob_text)
            
            # Clean up locations (remove spaces, uppercase)
            location_list = [re.sub(r'\s+', '', loc.upper()) for loc in location_matches]
            
            # --- STEP 4: Build Time Pairs ---
            pairs = []
            for j in range(len(time_matches) // 2):
                pairs.append((time_matches[j*2], time_matches[j*2+1]))
            if not pairs and len(time_matches) >= 2:
                pairs.append((time_matches[0], time_matches[1]))
            
            if not pairs:  # No time found, skip this entry
                continue
            
            # --- STEP 5: Assemble Course Entries ---
            for idx_pair, (start_time, end_time) in enumerate(pairs):
                course = {
                    'subject_code': subject_code,
                    'subject_name': current_subject_name,
                    'start_time': start_time,
                    'end_time': end_time,
                    'day': day_matches[idx_pair] if idx_pair < len(day_matches) else (day_matches[0] if day_matches else ''),
                    'location': location_list[idx_pair] if idx_pair < len(location_list) else (location_list[0] if location_list else '')
                }
                courses.append(course)
        
        return courses


In [4]:
# Helper: format output for display (kept simple)
def format_output(courses: List[Dict]) -> str:
    output = []
    output.append("=" * 80)
    output.append("EXTRACTED COURSE INFORMATION (Using KIE Predictor)")
    output.append("=" * 80)
    if not courses:
        output.append("\nNo courses found. This could mean:")
        output.append("  - The document format is different than expected")
        output.append("  - The image quality needs improvement")
        output.append("  - The model needs fine-tuning for this specific document type")
    for i, course in enumerate(courses, 1):
        output.append(f"\n[{i}] SUBJECT CODE : {course.get('subject_code','')}")
        if course.get('subject_name'):
            output.append(f"    SUBJECT NAME : {course.get('subject_name')}")
        output.append(f"    START TIME   : {course.get('start_time','')}")
        output.append(f"    END TIME     : {course.get('end_time','')}")
        if course.get('day'):
            output.append(f"    DAY          : {course.get('day')}")
        if course.get('location'):
            output.append(f"    LOCATION     : {course.get('location')}")
    output.append("\nTotal courses extracted: {}".format(len(courses)))
    return "\n".join(output)


In [5]:
# --- 1. CHANGE YOUR INPUT FILE HERE ---
INPUT_FILE_PATH = "img/reignCOR.pdf"  # <-- Testing with reignCOR.pdf


# --- 2. CHANGE YOUR OUTPUT FILE HERE ---
OUTPUT_JSON_PATH = "new/reignCOR_test.json" # <-- Put your JSON file path here


# --- Main Execution Logic ---

# 1. Initialize the extractor
extractor = CORExtractor(model)

# 2. Run the extraction
print(f"Starting extraction from '{INPUT_FILE_PATH}'...")
courses_list = extractor.extract_from_document(INPUT_FILE_PATH)
print(f"Found {len(courses_list)} course entries.")

# 3. Format the output for the JSON file
courses_dict = {}
for i, course in enumerate(courses_list, 1):
    courses_dict[f"sched{i}"] = course

# 4. Save the results to your output JSON file
with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(courses_dict, f, indent=2)

print(f"\nSuccessfully saved extracted data to '{OUTPUT_JSON_PATH}'")

# 5. (Optional) Print the formatted results
print("\n--- Console Output ---")
print(format_output(courses_list))

Starting extraction from 'img/reignCOR.pdf'...
Loading PDF at 300 DPI (higher quality)...
Found 8 course entries.

Successfully saved extracted data to 'new/reignCOR_test.json'

--- Console Output ---
EXTRACTED COURSE INFORMATION (Using KIE Predictor)

[1] SUBJECT CODE : BSAC125628
    SUBJECT NAME : STATISTICAL
    START TIME   : 02:30PM
    END TIME     : 04:00PM
    DAY          : MTH
    LOCATION     : CLA

[2] SUBJECT CODE : BSAC125655
    SUBJECT NAME : AUDITING
    START TIME   : 01:00PM
    END TIME     : 02:30PM
    DAY          : TF
    LOCATION     : CLA

[3] SUBJECT CODE : BSAC125676
    SUBJECT NAME : ACCOUNTING FOR
    START TIME   : 10:00AM
    END TIME     : 11:30AM
    LOCATION     : CLA

[4] SUBJECT CODE : BSAC125702
    SUBJECT NAME : UPDATES IN
    START TIME   : 07:00AM
    END TIME     : 08:30AM
    DAY          : S
    LOCATION     : CLA

[5] SUBJECT CODE : BSAC125861
    START TIME   : 10:00AM
    END TIME     : 11:30AM
    LOCATION     : CLA

[6] SUBJECT CODE :

In [6]:
# Test production ocr.py with the same file
import sys
sys.path.insert(0, '/home/sheer/Desktop/SchedScan/backend')

from api.utils.ocr import StudentCORExtractor

# Use the same model we already loaded
production_extractor = StudentCORExtractor(model)

# Test with same file
print("Testing production ocr.py on reignCOR.pdf...")
production_results = production_extractor.extract_from_document("img/reignCOR.pdf")
print(f"Production OCR found {len(production_results)} courses")

# Save production results
production_output_path = "new/reignCOR_production.json"
with open(production_output_path, 'w') as f:
    json.dump({f"sched{i+1}": course for i, course in enumerate(production_results)}, f, indent=2)
print(f"Saved production results to '{production_output_path}'")

# Display results
print("\n--- Production OCR Results ---")
for i, course in enumerate(production_results, 1):
    print(f"\n[{i}] SUBJECT CODE : {course.get('subject_code', 'N/A')}")
    print(f"    SUBJECT NAME : {course.get('subject_name', 'N/A')}")
    print(f"    START TIME   : {course.get('start_time', 'N/A')}")
    print(f"    END TIME     : {course.get('end_time', 'N/A')}")
    print(f"    DAY          : {course.get('day', 'N/A')}")
    print(f"    LOCATION     : {course.get('location', 'N/A')}")

Testing production ocr.py on reignCOR.pdf...
Production OCR found 8 courses
Saved production results to 'new/reignCOR_production.json'

--- Production OCR Results ---

[1] SUBJECT CODE : BSAC125628
    SUBJECT NAME : STATISTICAL
    START TIME   : 02:30PM
    END TIME     : 04:00PM
    DAY          : MTH
    LOCATION     : CLA

[2] SUBJECT CODE : BSAC125655
    SUBJECT NAME : AUDITING
    START TIME   : 01:00PM
    END TIME     : 02:30PM
    DAY          : TF
    LOCATION     : CLA

[3] SUBJECT CODE : BSAC125676
    SUBJECT NAME : ACCOUNTING FOR
    START TIME   : 10:00AM
    END TIME     : 11:30AM
    DAY          : 
    LOCATION     : CLA

[4] SUBJECT CODE : BSAC125702
    SUBJECT NAME : UPDATES IN
    START TIME   : 07:00AM
    END TIME     : 08:30AM
    DAY          : S
    LOCATION     : CLA

[5] SUBJECT CODE : BSAC125861
    SUBJECT NAME : 
    START TIME   : 10:00AM
    END TIME     : 11:30AM
    DAY          : 
    LOCATION     : CLA

[6] SUBJECT CODE : BSAC125864
    SUBJECT N

In [7]:
# Compare results between Notebook OCR and Production OCR
print("=" * 80)
print("COMPARISON: Notebook OCR vs Production OCR")
print("=" * 80)

# Read both JSON files
with open("new/reignCOR_test.json", 'r') as f:
    notebook_data = json.load(f)
    
with open("new/reignCOR_production.json", 'r') as f:
    production_data = json.load(f)

# Compare
if notebook_data == production_data:
    print("✅ RESULTS ARE IDENTICAL - Both OCR implementations produce the same output!")
else:
    print("❌ RESULTS DIFFER - Differences found:")
    for key in notebook_data:
        if key in production_data:
            if notebook_data[key] != production_data[key]:
                print(f"\n  {key}:")
                print(f"    Notebook:   {notebook_data[key]}")
                print(f"    Production: {production_data[key]}")
        else:
            print(f"\n  {key}: Only in notebook results")
    
    for key in production_data:
        if key not in notebook_data:
            print(f"\n  {key}: Only in production results")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Notebook OCR courses: {len(notebook_data)}")
print(f"Production OCR courses: {len(production_data)}")

# Show missing data summary
missing_days = sum(1 for k, v in production_data.items() if not v.get('day'))
missing_locations = sum(1 for k, v in production_data.items() if not v.get('location'))
missing_subject_names = sum(1 for k, v in production_data.items() if not v.get('subject_name'))

print(f"\nData completeness issues:")
print(f"  - Courses missing DAY: {missing_days}")
print(f"  - Courses missing LOCATION: {missing_locations}")
print(f"  - Courses missing SUBJECT NAME: {missing_subject_names}")

COMPARISON: Notebook OCR vs Production OCR
✅ RESULTS ARE IDENTICAL - Both OCR implementations produce the same output!

SUMMARY
Notebook OCR courses: 8
Production OCR courses: 8

Data completeness issues:
  - Courses missing DAY: 3
  - Courses missing LOCATION: 1
  - Courses missing SUBJECT NAME: 2


In [8]:
# Test with multiple files
test_files = [
    "img/reignCOR1.pdf",
    "img/reignCOR2.pdf", 
    "img/sheer2023COR.pdf"
]

print("=" * 80)
print("MULTI-FILE OCR ACCURACY TEST")
print("=" * 80)

for file_path in test_files:
    print(f"\n📄 Testing: {file_path}")
    print("-" * 60)
    
    try:
        # Test with notebook extractor
        notebook_ext = CORExtractor(model)
        notebook_results = notebook_ext.extract_from_document(file_path)
        
        # Test with production extractor  
        production_results = production_extractor.extract_from_document(file_path)
        
        # Compare
        notebook_dict = {f"sched{i+1}": c for i, c in enumerate(notebook_results)}
        production_dict = {f"sched{i+1}": c for i, c in enumerate(production_results)}
        
        if notebook_dict == production_dict:
            print(f"✅ MATCH - Both found {len(notebook_results)} courses")
        else:
            print(f"❌ MISMATCH")
            print(f"   Notebook: {len(notebook_results)} courses")
            print(f"   Production: {len(production_results)} courses")
            
        # Show completeness
        missing_days = sum(1 for c in production_results if not c.get('day'))
        missing_locs = sum(1 for c in production_results if not c.get('location'))
        missing_names = sum(1 for c in production_results if not c.get('subject_name'))
        
        print(f"   Missing days: {missing_days}, Missing locations: {missing_locs}, Missing names: {missing_names}")
        
    except Exception as e:
        print(f"❌ ERROR: {e}")

MULTI-FILE OCR ACCURACY TEST

📄 Testing: img/reignCOR1.pdf
------------------------------------------------------------
Loading PDF at 300 DPI (higher quality)...
✅ MATCH - Both found 2 courses
   Missing days: 0, Missing locations: 1, Missing names: 0

📄 Testing: img/reignCOR2.pdf
------------------------------------------------------------
Loading PDF at 300 DPI (higher quality)...
✅ MATCH - Both found 8 courses
   Missing days: 1, Missing locations: 4, Missing names: 0

📄 Testing: img/sheer2023COR.pdf
------------------------------------------------------------
Loading PDF at 300 DPI (higher quality)...
✅ MATCH - Both found 9 courses
   Missing days: 2, Missing locations: 4, Missing names: 2
